## Enabling Built-In Tools

# Lesson 4: Built-in Capabilities, Secure Permissions, and Tool Execution Lifecycles

In the previous lesson, you mastered system configurations using `ClaudeAgentOptions` to optimize model targeting, design system personas, and set reasoning turn limits. While these structures shape how an agent thinks and communicates, they do not grant the agent the ability to interact with external environments.

To expand your agent from a conversational companion into an active participant, you must use **Tools**. Tools allow the agent to read and write files, look up live web content, and execute shell configurations. In this module, you will learn how to enable built-in system tools, parse new streaming message block types representing tool operations, and configure directory rules and security permission modes.

---

## The Built-in Tool Architecture

The Claude Agent SDK provides a diverse ecosystem of out-of-the-box system tools to manage multi-layered developer operations and web scraping automation workflows.

While the engine features a robust toolkit, most production agentic patterns rely on **five core tools**:

| Tool Token | Functional Responsibility | Argument Input Parameters |
| --- | --- | --- |
| **`WebSearch`** | Performs external web search queries using active search endpoints, returning curated lists of titles, URLs, and descriptions. | `query` (string) |
| **`WebFetch`** | Ingests the raw, text-scraped layout of a specific URL, allowing the agent to parse documentation pages or code repos. | `url` (string), `prompt` (optional string constraint) |
| **`Bash`** | Launches an OS-level terminal shell instance inside the specified directory to run scripts, execute builds, or verify environments. | `command` (string), `description` (string rationale) |
| **`Read`** | Opens local disk files within the configured workspace bounds to extract text strings or inspect image data. | `file_path` (string) |
| **`Write`** | Creates new source code artifacts or overwrites existing file blocks directly onto the local filesystem disk. | `file_path` (string), `content` (string payload) |

---

## Explicit Registry Targeting via `allowed_tools`

To grant your agent instance access to these functions, declare them within the `allowed_tools` allowlist array inside your `ClaudeAgentOptions` configuration block. The agent can only see and execute the explicit tools you define, providing strict safety boundaries and predictable cost metrics:

```python
from claude_agent_sdk import ClaudeAgentOptions

options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    allowed_tools=[
        "WebSearch",  # Grant access to query search networks
        "WebFetch",   # Grant access to scrape URL string payloads
        "Bash",       # Grant access to run terminal shell configurations
        "Read",       # Grant access to open disk files
        "Write"       # Grant access to modify filesystem data
    ]
)

```

> 💡 **Self-Documenting Best Practice:** While enabling `WebSearch` automatically grants internal clearance for `WebFetch` (since searching without reading full pages is functionally incomplete), explicitly declaring `WebFetch` inside your `allowed_tools` collection keeps your codebase clean, transparent, and easy to maintain.

---

## Managing Tool Permissions via `permission_mode`

Every tool call needs explicit approval before the local runtime subprocess can execute the command on your system. To balance manual human authorization steps with headless script automation, configure the `permission_mode` parameter:

```python
options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    allowed_tools=["WebSearch", "WebFetch", "Bash", "Read"],
    # Automatically grant write approvals for file-editing operations
    permission_mode="acceptEdits" 
)

```

### Decoupled Permission Rule Exceptions:

When you configure `permission_mode="acceptEdits"`, the system applies an explicit override logic: the local runtime automatically grants execution authorization to any tool modifying localized code documents (like `Write` and `Edit`).

Because this rule handles file modifications automatically, **the agent can successfully generate or edit files even if you omit `"Write"` from the core `allowed_tools` register list.**

---

## Enforcing Filesystem Boundaries via `cwd`

Before letting an autonomous agent execute terminal loops or modify repository scripts, you must define its working boundaries. The `cwd` (Current Working Directory) parameter dictates where the underlying Claude Code tool engine starts and operates.

By default, tool execution paths follow the directory where you *invoked* Python, not necessarily where your running script file resides. To prevent relative paths from breaking when scripts run from different folders, use `Path(__file__)` to bind the agent's workspace directly to the script location:

```python
from pathlib import Path
from claude_agent_sdk import ClaudeAgentOptions

options = ClaudeAgentOptions(
    model="haiku",
    max_turns=10,
    allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write"],
    permission_mode="acceptEdits",
    # Bind the agent's filesystem context strictly to this script's directory
    cwd=Path(__file__).parent 
)

```

### Runtime Path Resolution Invariants:

* **Terminal Shell Traversal:** Local execution blocks run as if you had issued a change-directory command (`cd <cwd>`) before running the script.
* **Relative vs. Absolute Paths:** Relative tool declarations (e.g., `summaries/data.md`) map safely from the configured `cwd` node. However, if Claude passes an absolute path (`/workspace/summaries/data.md`), the file path resolves starting directly from your machine's filesystem root, bypassing the localized `cwd` rule.

---

## Deep Dive: Streaming Block Abstractions for Tools

When you enable tool utilities, the asynchronous message stream produced by the `query()` generator introduces two specialized, strongly-typed block layouts within your user and assistant communication loops:

```text
                                STREAMING TIMELINE FLOW
────────────────────────────────────────────────────────────────────────────────────────
[AssistantMessage] ──► Yields ToolUseBlock   (Agent declares intent + passes arguments)
                             │
                             ▼  [Subprocess Executes Local Operation Tool]
                             │
[UserMessage]      ──► Yields ToolResultBlock (Runtime returns raw result data string)

```

### 1. `ToolUseBlock` (Found inside `AssistantMessage.content`)

Indicates that Claude has completed a reasoning cycle and decided to call a specific tool. It holds the tool invocation metadata but *does not* contain the tool's execution result.

* **`block.name`:** The string token identifier matching the registered tool name (e.g., `"Bash"`).
* **`block.input`:** A Python dictionary containing the exact argument parameters generated by Claude (e.g., `{"command": "mkdir -p summaries"}`).
* **`block.id`:** A unique tracking hash matching this tool transaction block across the loop.

### 2. `ToolResultBlock` (Found inside `UserMessage.content`)

Emitted automatically after the local runtime completes the task. It passes the raw system results back into the conversation context so the agent can review the output.

* **`block.tool_use_id`:** The cross-reference tracking identifier linking this result back to its parent `ToolUseBlock`.
* **`block.content`:** The raw string payload returned by your operating system (e.g., directory listings, fetch outputs, or validation text).
* **`block.is_error`:** A boolean flag tracking whether the tool function executed successfully or raised a system error.

---

## Inspecting Interleaved Tool Streams and Logs

To audit your agent's live reasoning loops, check for both `ToolUseBlock` and `ToolResultBlock` instances inside your async streaming handler. The following script shows a clean pattern for capturing and printing these transitions:

```python
from claude_agent_sdk import query, AssistantMessage, UserMessage, TextBlock, ToolUseBlock, ToolResultBlock

async def run_monitored_agent():
    prompt = "Search for the Claude Agent SDK and save a brief summary to ./summaries/sdk.md"
    
    async for message in query(prompt=prompt, options=options):
        # 1. Catch Agent Decision Turns
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 [Claude thought]: {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool Invocation]: {block.name}")
                    for param_key, param_val in (block.input or {}).items():
                        # Truncate large tool payloads to keep terminal output readable
                        print(f"  -> {param_key}: {str(param_val)[:40]}...")
                        
        # 2. Catch System Execution Result Transitions
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Output Block Received]")
                    if isinstance(block.content, str) and block.content.strip():
                        print(f"  -> Data Trace: {block.content.strip()[:80]}...")
                    else:
                        print("  -> Execution Complete (Silent Success Signature)")

```

---

## Step-by-Step Diagnostic Execution Trace

When running this code, watch the terminal closely. You will see a multi-turn autonomous execution workflow unfold as the agent gathers info and modifies your workspace:

### Turn 1: Information Gathering (`WebSearch`)

The agent realizes it lacks documentation context for the Claude Agent SDK and triggers a web search:

```text
💬 [Claude thought]: I need to gather info on the Claude Agent SDK before generating a summary. I will start with a web lookup.

🔧 [Tool Invocation]: WebSearch
  -> query: Claude Agent SDK documentation...

✅ [Tool Output Block Received]
  -> Data Trace: Links: [{"title": "Overview - Claude Agent SDK Docs", "url": "https://docs.claude.com..."}]...

```

### Turn 2: Scraping Details (`WebFetch`)

The search returns a reference link. The agent reads the page layout to capture technical details:

```text
💬 [Claude thought]: I found the official documentation link. Now I'll extract the core specifications.

🔧 [Tool Invocation]: WebFetch
  -> url: https://docs.claude.com/en/overview...

✅ [Tool Output Block Received]
  -> Data Trace: # Claude Agent SDK Overview\nThis library exposes programmatic bindings...

```

### Turn 3: Defensive File Check (`Bash`)

With the data saved in its context, the agent prepares to write the file. It checks if the target directory exists:

```text
💬 [Claude thought]: I have the required data points. Let me verify if the summaries directory is available.

🔧 [Tool Invocation]: Bash
  -> command: ls -la ./summaries 2>/dev/null...
  -> description: Check for target folder directory presence...

✅ [Tool Output Block Received]
  -> Data Trace: Directory does not exist...

```

### Turn 4: Directory Creation (`Bash`)

The system returns a missing path signature. The agent adapts its plan and creates the missing directory:

```text
💬 [Claude thought]: The folder is missing. I will create it using mkdir before writing the file.

🔧 [Tool Invocation]: Bash
  -> command: mkdir -p ./summaries...
  -> description: Create missing summaries directory folder...

✅ [Tool Output Block Received]
  -> Execution Complete (Silent Success Signature)

```

### Turn 5: File Generation (`Write`)

With the path initialized, the agent uses the write tool to save the markdown summary to disk:

```text
🔧 [Tool Invocation]: Write
  -> file_path: ./summaries/sdk.md...
  -> content: # Claude Agent SDK Summary\n## Features...

✅ [Tool Output Block Received]
  -> Data Trace: File created successfully at: ./summaries/sdk.md...

```

### Turn 6: Task Summary & Resolution

The file modification is complete. The agent reviews the terminal response and wraps up the session:

```text
💬 [Claude thought]: Perfect! I have successfully generated your documentation asset at `./summaries/sdk.md`. The file contains clear breakdowns of context management, tool calling, and anyio lifecycles. Let me know if you need any adjustments!

```

---

## Summary

You have unlocked the ability to build active, tool-enabled agents using the Claude Agent SDK:

* The **`allowed_tools`** parameter acts as a secure, fine-grained allowlist for controlling what operations your agent can see.
* The **`permission_mode`** parameter balances execution safety with automated speed (e.g., using `acceptEdits` to auto-approve file changes).
* The **`cwd`** parameter sets strict filesystem boundaries for all local actions.
* The **`ToolUseBlock`** and **`ToolResultBlock`** types allow your code to monitor, log, and trace complex multi-turn workflows in real time.

Let's head into the practice labs to configure and launch your first tool-enabled agent! 🚀

## Observing Agent Tool Calls in Action

In a previous task, you saw that the agent couldn't complete tasks without tools enabled. Now you'll see what was happening behind the scenes — the actual tool call attempts the agent was making.

The code has no tools enabled in ClaudeAgentOptions. When you ask the agent to search the web and save a file, it will attempt to call tools, and these attempts appear as ToolUseBlock objects in the message stream — even though they can't execute.

Right now, your code only displays TextBlock objects, so you can only see the agent's text responses. The tool call attempts are happening but invisible.

Your task is to make these attempted tool calls visible by adding code to display ToolUseBlock objects:

    Add an elif block that checks if a content block is a ToolUseBlock
    Print the tool name with the 🔧 emoji
    Display the input parameters

Once complete, you'll see the agent attempting to call WebSearch, Bash, and Write — the same tools that were unavailable before, but now you can observe exactly what the agent was trying to do!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def main():
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10
    )

    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # TODO: Add an elif to check if block is a ToolUseBlock
                    # TODO: Print the tool name with the 🔧 emoji (use f"\n🔧 [Tool: {block.name}]")
                    # TODO: Check if block.input exists
                        # TODO: Iterate through block.input.items() to get key-value pairs
                            # TODO: Print each parameter with its value


if __name__ == "__main__":
    anyio.run(main)

```

Here is the completed code with the `elif` block added to inspect and print the `ToolUseBlock` data inside the `AssistantMessage.content` loop:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
)


async def main():
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10
    )

    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # Add an elif to check if block is a ToolUseBlock
                elif isinstance(block, ToolUseBlock):
                    # Print the tool name with the 🔧 emoji
                    print(f"\n🔧 [Tool: {block.name}]")
                    # Check if block.input exists
                    if block.input:
                        # Iterate through block.input.items() to get key-value pairs
                        for key, value in block.input.items():
                            # Print each parameter with its value
                            print(f"  {key}: {value}")


if __name__ == "__main__":
    anyio.run(main)

```

### What You Will Observe:

When you run this code, even though the tools are disabled in `ClaudeAgentOptions`, you will now see **Claude's intent** appear in the console log stream.

You'll see output like `🔧 [Tool: WebSearch]` with its `query` string parameters printed out. This proves that while the AI engine recognizes it *needs* to look up information or write a file and structures the correct payloads, the underlying runtime platform is intercepting and restricting execution until you provide explicit access!

## Displaying Tool Execution Results

In the previous exercise, you saw your agent attempting to call tools through ToolUseBlock objects. Now it's time to see what comes back from those tool calls.

After the agent requests a tool, the system sends back the results in a UserMessage containing ToolResultBlock objects. Since we haven't enabled any tools yet, you'll see error messages indicating the tools aren't available.

Right now, your code only handles AssistantMessage instances. Your task is to add code that handles UserMessage instances and displays their ToolResultBlock content:

    Check if each message is a UserMessage
    Loop through its content blocks
    Find any ToolResultBlock instances
    Display the output (show first 100 chars for non-empty content, or "(completed successfully)" for empty results)

Follow the same pattern as the AssistantMessage handling above — just check for different message and block types. Once complete, you'll see both what the agent requests AND what it receives back!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")
                    
        # TODO: Check if the incoming message is a UserMessage
            # TODO: Iterate through the content blocks in the message
                # TODO: Check if the current block is a ToolResultBlock
                    # TODO: Print a header indicating this is a tool result
                    # TODO: Check if the block content is a string type
                        # TODO: Strip whitespace from the content
                        # TODO: If content is not empty, show a preview (first 100 chars)
                        # TODO: If content is empty, indicate successful completion


if __name__ == "__main__":
    anyio.run(main)
```

Here is the completed code with the tracking block added to handle incoming `UserMessage` streams and extract previews from `ToolResultBlock` payloads.

This mirrors your assistant parsing logic, allowing you to trace exactly what error context or results the system feeds back to the model after a tool is called:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")
                    
        # Check if the incoming message is a UserMessage
        elif isinstance(message, UserMessage):
            # Iterate through the content blocks in the message
            for block in message.content:
                # Check if the current block is a ToolResultBlock
                if isinstance(block, ToolResultBlock):
                    # Print a header indicating this is a tool result
                    print(f"\n✅ [Tool Result]")
                    # Check if the block content is a string type
                    if isinstance(block.content, str):
                        # Strip whitespace from the content
                        content = block.content.strip()
                        # If content is not empty, show a preview (first 100 chars)
                        if content:
                            print(f"  {content[:100]}...")
                        # If content is empty, indicate successful completion
                        else:
                            print("  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)

```

### What You Will See in Your Logs:

Because tools are not listed in your `allowed_tools` register yet, running this script will produce a clean trace of a blocked operation cycle:

1. First, Claude emits a `ToolUseBlock` detailing its intent to use `WebSearch`.
2. The runtime engine blocks this action and injects a `ToolResultBlock` nested inside a `UserMessage`.
3. Your new logic catches this and prints the response, which will explicitly display a restriction notification like: `Error: Tool WebSearch is not allowed...`. This feedback helps you visually trace the loop before turning on full production permissions!

## Unlocking Your Agent's Action Potential

In previous exercises, you saw how the agent tries to call tools and describes what it would do, but without any tools enabled, it can't actually perform any actions. Now, you'll finally unlock real tool use for your agent.

Your task:
Update the ClaudeAgentOptions configuration to include the allowed_tools parameter, and provide a list containing these five core tools:

    "WebSearch" — to search the internet
    "WebFetch" — to retrieve content from URLs
    "Bash" — to execute shell commands
    "Read" — to read files
    "Write" — to create and modify files

With these tools enabled, your agent will be able to take real actions—searching, reading, writing, and running commands—rather than just talking about them. This simple change transforms your agent from a passive explainer into an active, autonomous assistant!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # TODO: Add the allowed_tools parameter with a list containing: "WebSearch", "WebFetch", "Bash", "Read", "Write"
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")  # Show first 30 chars
                    
        # ToolResultBlock contains the output from tool execution
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Result]")
                    if isinstance(block.content, str):
                        content = block.content.strip()
                        if content:
                            # Show preview of non-empty content
                            print(f"  {content[:100]}...")
                        else:
                            # Indicate successful execution with no output
                            print(f"  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)

```

Here is the updated code with the `allowed_tools` registry fully configured inside your `ClaudeAgentOptions` instance.

By adding this explicit allowlist array, you give the local agent runtime permissions to execute web lookups, scrape URL contexts, and interact with the local filesystem:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # Add the allowed_tools parameter with a list containing: "WebSearch", "WebFetch", "Bash", "Read", "Write"
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10,
        allowed_tools=["WebSearch", "WebFetch", "Bash", "Read", "Write"]
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt="Search for the Claude Agent SDK and save a brief summary to ./summaries/claude-agent-sdk.md",
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")  # Show first 30 chars
                    
        # ToolResultBlock contains the output from tool execution
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Result]")
                    if isinstance(block.content, str):
                        content = block.content.strip()
                        if content:
                            # Show preview of non-empty content
                            print(f"  {content[:100]}...")
                        else:
                            # Indicate successful execution with no output
                            print(f"  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)

```

### Architectural Shifts to Expect:

* **True Multi-Turn Workflows:** Instead of failing on the first step with a permission error block, you will now see the agent complete its entire workflow loop. It will execute a `WebSearch`, read page parameters with `WebFetch`, check and set path hierarchies with `Bash`, and generate your target markdown resource via `Write` dynamically!
* **Defensive Scoping:** Even with these options active, the local execution environment remains constrained to protect your global filesystem. Operations resolve securely relative to the application's runtime context.

## Granting Broader Tool Permission

By default, your agent can only use the tools you specify in the allowed_tools list. However, you can give your agent access to additional tools—such as Edit for modifying existing files—without explicitly adding them to allowed_tools. To do this, set the permission_mode parameter in your ClaudeAgentOptions to "acceptEdits".

With permission_mode="acceptEdits", the agent is automatically allowed to use any tool that edits files, even if those tools aren’t listed in allowed_tools. This expands the agent’s capabilities, letting it perform file modifications using tools like Edit and others that require elevated permissions.

Your task is to update your agent configuration by adding permission_mode="acceptEdits" to your ClaudeAgentOptions. Then, run your code with the provided prompt and observe how the agent is now able to use advanced editing tools to modify files directly.

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10,
        allowed_tools=[
            "WebSearch",  # Search the web for information
            "WebFetch",   # Fetch full content from URLs (auto-enabled with WebSearch)
            "Bash",       # Execute shell commands
            "Read",       # Read files from disk
            "Write",      # Create/write files
        ],
        # TODO: Add permission_mode parameter set to "acceptEdits" to auto-approve edit tools
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt=(
            "Add a new section called 'Getting Started' to the already existing " 
            "summaries/claude-agent-sdk.md with instructions on how to install the SDK using pip."
        ),
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")  # Show first 30 chars
                    
        # ToolResultBlock contains the output from tool execution
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Result]")
                    if isinstance(block.content, str):
                        content = block.content.strip()
                        if content:
                            # Show preview of non-empty content
                            print(f"  {content[:100]}...")
                        else:
                            # Indicate successful execution with no output
                            print(f"  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)
```

Here is the completed code with `permission_mode="acceptEdits"` successfully integrated into your `ClaudeAgentOptions` configuration block.

By defining this permission parameter, you grant the runtime environment explicit authority to auto-approve file-editing operations (such as using advanced tools like `Edit`), allowing the agent to modify the existing summary document cleanly:

```python
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():
    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10,
        allowed_tools=[
            "WebSearch",  # Search the web for information
            "WebFetch",   # Fetch full content from URLs (auto-enabled with WebSearch)
            "Bash",       # Execute shell commands
            "Read",       # Read files from disk
            "Write",      # Create/write files
        ],
        # Add permission_mode parameter set to "acceptEdits" to auto-approve edit tools
        permission_mode="acceptEdits"
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt=(
            "Add a new section called 'Getting Started' to the already existing " 
            "summaries/claude-agent-sdk.md with instructions on how to install the SDK using pip."
        ),
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")  # Show first 30 chars
                    
        # ToolResultBlock contains the output from tool execution
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Result]")
                    if isinstance(block.content, str):
                        content = block.content.strip()
                        if content:
                            # Show preview of non-empty content
                            print(f"  {content[:100]}...")
                        else:
                            # Indicate successful execution with no output
                            print(f"  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)

```

### What to Observe in the Stream:

When running this task, Claude won't overwrite the entire file with a standard `Write` block. Instead, it can now seamlessly use advanced file tools like `Edit` to target the specific file lines and append the new `"Getting Started"` section, making file adjustments highly robust, safe, and clean!

## Setting the Agent Working Directory

Now that you understand how tools transform agents from text responders into active assistants, it's time to configure where your agent starts working from.

Your task has two parts:

    Add the cwd parameter to the ClaudeAgentOptions configuration and set it to Path(__file__).parent / "summaries"
    Update the prompt to reference just claude-agent-sdk.md instead of summaries/claude-agent-sdk.md

By setting the working directory to the summaries folder, relative paths in your prompts start from there — meaning you can use simplified filenames instead of full paths. This demonstrates how cwd controls where the agent's file operations begin.

Once you make these changes, run the code and watch as the agent searches for information and updates the markdown file using the simplified path!

```
import anyio
from pathlib import Path
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
    ToolUseBlock,
    UserMessage,
    ToolResultBlock
)


async def main():

    # Configure the agent with ClaudeAgentOptions
    options = ClaudeAgentOptions(
        model="haiku",
        system_prompt="You are an enthusiastic beginner friendly tutor.",
        max_turns=10,
        allowed_tools=[
            "WebSearch",
            "WebFetch",
            "Bash",
            "Read",
            "Write",
        ],
        permission_mode="acceptEdits",
        # TODO: Set the working directory to the summaries subdirectory using Path(__file__).parent / "summaries"
    )
    
    # Pass options to query() to apply configuration
    async for message in query(
        prompt=(
            "Add a new section called 'Getting Started' to the already existing " 
            "summaries/claude-agent-sdk.md with instructions on how to install the SDK using pip."
        ),
        options=options
    ):
        
        # Display assistant messages
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"\n💬 Claude Response:")
                    print(block.text)
                # ToolUseBlock represents the agent calling a tool
                elif isinstance(block, ToolUseBlock):
                    print(f"\n🔧 [Tool: {block.name}]")
                    if block.input:
                        for key, value in block.input.items():
                            print(f"  {key}: {str(value)[:30]}...")  # Show first 30 chars
                    
        # ToolResultBlock contains the output from tool execution
        elif isinstance(message, UserMessage):
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"\n✅ [Tool Result]")
                    if isinstance(block.content, str):
                        content = block.content.strip()
                        if content:
                            # Show preview of non-empty content
                            print(f"  {content[:100]}...")
                        else:
                            # Indicate successful execution with no output
                            print(f"  (completed successfully)")


if __name__ == "__main__":
    anyio.run(main)
```